In [1]:
import os
import glob
import zipfile

zip_files = glob.glob("/content/*.zip")

if not zip_files:
    raise FileNotFoundError("No ZIP file was uploaded.")

zip_path = zip_files[0]
extract_folder = "/content/titanic_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(extract_folder)

print("Dataset extracted successfully.")
print("Extracted files:")

for root, folders, files in os.walk(extract_folder):
    for file in files:
        print(os.path.join(root, file))

Dataset extracted successfully.
Extracted files:
/content/titanic_data/gender_submission.csv
/content/titanic_data/train.csv
/content/titanic_data/test.csv


In [2]:
import pandas as pd
import glob

train_files = glob.glob(
    "/content/titanic_data/**/train.csv",
    recursive=True
)

test_files = glob.glob(
    "/content/titanic_data/**/test.csv",
    recursive=True
)

if not train_files:
    raise FileNotFoundError("train.csv was not found.")

if not test_files:
    raise FileNotFoundError("test.csv was not found.")

train_path = train_files[0]
test_path = test_files[0]

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train path:", train_path)
print("Test path:", test_path)

print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

Train path: /content/titanic_data/train.csv
Test path: /content/titanic_data/test.csv
Training data shape: (891, 12)
Testing data shape: (418, 11)


In [3]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
print("Training columns:")
print(train_df.columns.tolist())

print("\nMissing values:")
print(train_df.isnull().sum())

Training columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [5]:
def create_features(data):
    data = data.copy()

    data["Title"] = data["Name"].str.extract(
        r",\s*([^.]*)\.",
        expand=False
    ).str.strip()

    rare_titles = [
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona"
    ]

    data["Title"] = data["Title"].replace(
        rare_titles,
        "Rare"
    )

    data["Title"] = data["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    data["FamilySize"] = (
        data["SibSp"] +
        data["Parch"] +
        1
    )

    data["IsAlone"] = (
        data["FamilySize"] == 1
    ).astype(int)

    return data


train_processed = create_features(train_df)
test_processed = create_features(test_df)

train_processed[
    ["Name", "Title", "FamilySize", "IsAlone"]
].head()

,Name,Title,FamilySize,IsAlone
0,"Braund, Mr. Owen Harris",Mr,2,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0
2,"Heikkinen, Miss. Laina",Miss,1,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0
4,"Allen, Mr. William Henry",Mr,1,1


In [6]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Title",
    "FamilySize",
    "IsAlone"
]

X = train_processed[features]
y = train_processed["Survived"]

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone
0,3,male,22.0,1,0,7.2500,S,Mr,2,0
1,1,female,38.0,1,0,71.2833,C,Mrs,2,0
2,3,female,26.0,0,0,7.9250,S,Miss,1,1
3,1,female,35.0,1,0,53.1000,S,Mrs,2,0
4,3,male,35.0,0,0,8.0500,S,Mr,1,1


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title"
]

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

print("Model pipeline created successfully.")

Model pipeline created successfully.


In [8]:
pipeline.fit(X, y)

print("Final model trained successfully.")

Final model trained successfully.


In [9]:
X_test = test_processed[features]

print("Test data shape:", X_test.shape)

Test data shape: (418, 10)


In [10]:
test_predictions = pipeline.predict(X_test)

print("Predictions:")
print(test_predictions[:20])

Predictions:
[0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 0 1]


In [11]:
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions
})

submission.to_csv(
    "/content/submission.csv",
    index=False
)

print("submission.csv created successfully!")

submission.csv created successfully!


In [12]:
print(submission.head(10))
print("\nShape:", submission.shape)

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

Shape: (418, 2)
